In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Wan initial-noise inversion: fixed validation

Run all executes 2 contents x 2 seeds x OFF/A/B = 12 videos, each with 181
frames at 320x512 and 8 fps. No mode or scan is needed. New outputs, source
archive and launcher/child logs persist under
`MyDrive/Video-WM/VideoInversion/video_inversion_<UTC>`.

The frozen method writes 16 complementary A/B payload bits into channel0
initial-noise signs using a keyed coordinate permutation and sign pad. Each bit
has 160 repetitions per latent slice across 46 slices: 7360 correlated votes.
Other channels and initial magnitudes are unchanged. Zero/ties are erasures.

Generation retains the native 50-step UniPC forward path. Reception reads the
actual saved CRF18 yuv420p MP4, VAE-encodes it and applies one 50-step explicit
flow Euler approximate inversion. Reversed actual sigma nodes are paired with
original integer times [0,t49,...,t1]. This is not an exact UniPC inverse.
Known original prompt/negative/CFG5 are public receiver conditions. Receiver
seed0 loader noise is discarded; writer noise/terminal never reconstruct the
receiver. Truth and initial-noise comparisons are reporting-only afterward.

Generation, media and receiver have separate model lifetimes. Maximum calls:
2400 Transformer forwards, 600 native forward steps, 600 inverse updates,
12 VAE decodes, 12 VAE encodes and 12 MP4 saves. Fixed cases, arms and 46 latent
slice rows retain failures. Inspect result.json and logs even when a cell fails.
Recovery, same-case OFF MP4 quality and execution completion are separate;
there is no scientific PASS, FPR claim or invented quality tolerance. Full-space
redundancy is not a strictly fair carrier comparison against GROW.

The reused torch2.11.0+cu128/diffusers0.40.0 setup restores dependencies if needed
and verifies them in a fresh subprocess. Matching torch is not reinstalled.
Select a GPU runtime and Run all. No GPU model-name restriction is imposed.
This notebook has static and mocked-launcher validation only; the assistant has
not executed pretrained models, GPU, Colab or Drive experiments.

Source SHA: a4bcd1b4b07060a69e2bd2233e2c95e64c314c22.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = 'a4bcd1b4b07060a69e2bd2233e2c95e64c314c22'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'video_inversion_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/VideoInversion') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.video_inversion_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({'status': result['status'], 'video_denominator': result['video_denominator'], 'case_denominator': result['case_denominator'], 'fixed_calls': result['fixed_calls'], 'actual_calls_observed': result.get('actual_calls_observed'), 'call_count_case_coverage': result.get('call_count_case_coverage'), 'cases': {k: v['status'] for k, v in result['cases'].items()}}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
